In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# helper to load and summarize generic telemetry files
def summarize_file(path, id_col='cmdb_id', name_col=None, name_limit=50, id_limit=20):
    df = pd.read_csv(path)
    # ensure timestamp exists
    if 'timestamp' in df.columns and not df['timestamp'].isnull().all():
        times = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        tmin = times.min()
        tmax = times.max()
        tmin_s = None if pd.isna(tmin) else tmin.strftime('%Y-%m-%d %H:%M:%S%z')
        tmax_s = None if pd.isna(tmax) else tmax.strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        tmin_s = tmax_s = None

    total_rows = int(df.shape[0])
    unique_ids = sorted(df[id_col].dropna().unique().astype(str)) if id_col in df.columns else []
    unique_ids_preview = unique_ids[:id_limit]
    unique_id_count = len(unique_ids)

    names_preview = None
    unique_names_count = 0
    if name_col and name_col in df.columns:
        unique_names = sorted(df[name_col].dropna().unique().astype(str))
        unique_names_count = len(unique_names)
        names_preview = unique_names[:name_limit]

    summary = {
        'total_rows': total_rows,
        'unique_cmdb_id_count': unique_id_count,
        'unique_cmdb_id_preview_up_to_20': unique_ids_preview,
        'kpi_trace_log_names_count': unique_names_count if name_col else None,
        'kpi_trace_log_names_preview_up_to_50': names_preview,
        'min_timestamp_as_Asia/Shanghai': tmin_s,
        'max_timestamp_as_Asia/Shanghai': tmax_s,
    }
    return df, summary

# Process metric.csv
metric_path = Path('metric.csv')
df_metric, metric_summary = summarize_file(metric_path, id_col='cmdb_id', name_col='kpi_name', name_limit=50, id_limit=20)

# Process trace.csv
trace_path = Path('trace.csv')
df_trace, trace_summary = summarize_file(trace_path, id_col='cmdb_id', name_col='trace_name', name_limit=50, id_limit=20)

# Process log.csv
log_path = Path('log.csv')
df_log, log_summary = summarize_file(log_path, id_col='cmdb_id', name_col='log_name', name_limit=50, id_limit=20)

# Process error_logs.csv with message-specific handling
error_path = Path('error_logs.csv')
df_err = pd.read_csv(error_path)
# timestamp handling
if 'timestamp' in df_err.columns and not df_err['timestamp'].isnull().all():
    times_err = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    err_tmin = times_err.min().strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(times_err.min()) else None
    err_tmax = times_err.max().strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(times_err.max()) else None
else:
    err_tmin = err_tmax = None

err_total_rows = int(df_err.shape[0])
err_unique_ids = sorted(df_err['cmdb_id'].dropna().unique().astype(str)) if 'cmdb_id' in df_err.columns else []
err_unique_ids_preview = err_unique_ids[:20]
err_unique_id_count = len(err_unique_ids)

message_present = 'message' in df_err.columns
message_samples = None
if message_present:
    distinct_msgs = df_err['message'].dropna().astype(str).unique()
    message_samples = list(distinct_msgs[:5])

error_summary = {
    'total_rows': err_total_rows,
    'unique_cmdb_id_count': err_unique_id_count,
    'unique_cmdb_id_preview_up_to_20': err_unique_ids_preview,
    'message_column_present': message_present,
    'message_samples_up_to_5_distinct': message_samples,
    'min_timestamp_as_Asia/Shanghai': err_tmin,
    'max_timestamp_as_Asia/Shanghai': err_tmax,
}

summaries = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_summary,
}

summaries

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n\n"
    "metric.csv: 144,252 rows; 59 unique cmdb_id (sample up to 20: "
    "adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, "
    "cartservice, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, "
    "checkoutservice, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, "
    "currencyservice, currencyservice-0, currencyservice-1, currencyservice-2). "
    "1152 unique KPI names (examples: app.grpc.count, app.http.count, many container.* metrics). "
    "Timestamps: 2022-03-20 16:30:00+0800 to 2022-03-20 16:59:00+0800.\n\n"
    "trace.csv: 36,768 rows; 40 unique cmdb_id (sample up to 20 includes many service pods such as adservice-0..2, cartservice-0..2, checkoutservice-0..2, currencyservice-0..2, emailservice-0..2, frontend-0..2). "
    "200 unique trace names (examples: trace.from_<service>.<pod>.duration_mean/_p95/_error_rate/_row_count). "
    "Timestamps: 2022-03-20 16:30:00+0800 to 2022-03-20 16:59:00+0800.\n\n"
    "log.csv: 1,814 rows; 31 unique cmdb_id (sample up to 20 includes various pods such as adservice-0..2, cartservice-0..2, checkoutservice-0..2, currencyservice-0..2, emailservice-0..2, frontend-0..2, paymentservice-0..1). "
    "2 unique log names: log.error_count, log.row_count. "
    "Timestamps: 2022-03-20 16:30:00+0800 to 2022-03-20 16:59:00+0800.\n\n"
    "error_logs.csv: 2,007 rows; 3 unique cmdb_id: frontend-0, frontend-1, frontend-2. "
    "The 'message' column is present. Up to 5 distinct sample messages (one shown): "
    "\"severity: warning, message: failed to retrieve ads\". "
    "Timestamps: 2022-03-20 16:30:00+0800 to 2022-03-20 16:59:58+0800."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 144252, 'unique_cmdb_id_count': 59, 'unique_cmdb_id_preview_up_to_20': [np.str_('adservice'), np.str_('adservice-0'), np.str_('adservice-1'), np.str_('adservice-2'), np.str_('adservice2'), np.str_('adservice2-0'), np.str_('cartservice'), np.str_('cartservice-0'), np.str_('cartservice-1'), np.str_('cartservice-2'), np.str_('cartservice2-0'), np.str_('checkoutservice'), np.str_('checkoutservice-0'), np.str_('checkoutservice-1'), np.str_('checkoutservice-2'), np.str_('checkoutservice2-0'), np.str_('currencyservice'), np.str_('currencyservice-0'), np.str_('currencyservice-1'), np.str_('currencyservice-2')], 'kpi_trace_log_names_count': 1152, 'kpi_trace_log_names_preview_up_to_50': [np.str_('app.grpc.count'), np.str_('app.grpc.mrt'), np.str_('app.grpc.rr'), np.str_('app.grpc.sr'), np.str_('app.http.count'), np.str_('app.http.mrt'), np.str_('app.http.rr'), np.str_('app.http.sr'), np.str_('container.node-5.container_cpu_cfs_periods'), np.str_('container.node-5.container_cpu_cfs_throttled_periods'), np.str_('container.node-5.container_cpu_cfs_throttled_seconds'), np.str_('container.node-5.container_cpu_load_average_10s'), np.str_('container.node-5.container_cpu_system_seconds'), np.str_('container.node-5.container_cpu_usage_seconds'), np.str_('container.node-5.container_cpu_user_seconds'), np.str_('container.node-5.container_file_descriptors'), np.str_('container.node-5.container_fs_inodes./dev/vda1'), np.str_('container.node-5.container_fs_inodes_free./dev/vda1'), np.str_('container.node-5.container_fs_io_current./dev/vda1'), np.str_('container.node-5.container_fs_io_time_seconds./dev/vda1'), np.str_('container.node-5.container_fs_io_time_weighted_seconds./dev/vda1'), np.str_('container.node-5.container_fs_limit_MB./dev/vda1'), np.str_('container.node-5.container_fs_read_seconds./dev/vda1'), np.str_('container.node-5.container_fs_reads./dev/vda'), np.str_('container.node-5.container_fs_reads./dev/vda1'), np.str_('container.node-5.container_fs_reads_MB./dev/vda'), np.str_('container.node-5.container_fs_reads_merged./dev/vda1'), np.str_('container.node-5.container_fs_sector_reads./dev/vda1'), np.str_('container.node-5.container_fs_sector_writes./dev/vda1'), np.str_('container.node-5.container_fs_usage_MB./dev/vda1'), np.str_('container.node-5.container_fs_write_seconds./dev/vda1'), np.str_('container.node-5.container_fs_writes./dev/vda'), np.str_('container.node-5.container_fs_writes./dev/vda1'), np.str_('container.node-5.container_fs_writes_MB./dev/vda'), np.str_('container.node-5.container_fs_writes_merged./dev/vda1'), np.str_('container.node-5.container_last_seen'), np.str_('container.node-5.container_memory_cache'), np.str_('container.node-5.container_memory_failcnt'), np.str_('container.node-5.container_memory_failures.container.pgfault'), np.str_('container.node-5.container_memory_failures.container.pgmajfault'), np.str_('container.node-5.container_memory_failures.hierarchy.pgfault'), np.str_('container.node-5.container_memory_failures.hierarchy.pgmajfault'), np.str_('container.node-5.container_memory_mapped_file'), np.str_('container.node-5.container_memory_max_usage_MB'), np.str_('container.node-5.container_memory_rss'), np.str_('container.node-5.container_memory_swap'), np.str_('container.node-5.container_memory_usage_MB'), np.str_('container.node-5.container_memory_working_set_MB'), np.str_('container.node-5.container_network_receive_MB.eth0'), np.str_('container.node-5.container_network_receive_errors.eth0')], 'min_timestamp_as_Asia/Shanghai': '2022-03-20 16:30:00+0800', 'max_timestamp_as_Asia/Shanghai': '2022-03-20 16:59:00+0800'}, 'trace.csv': {'total_rows': 36768, 'unique_cmdb_id_count': 40, 'unique_cmdb_id_preview_up_to_20': [np.str_('adservice-0'), np.str_('adservice-1'), np.str_('adservice-2'), np.str_('adservice2-0'), np.str_('cartservice-0'), np.str_('cartservice-1'), np.str_('cartservice-2'), np.str_('cartservice2-0'), np.str_('checkoutservice-0'), np.str_('checkoutservice-1'), np.str_('checkoutservice-2'), np.str_('checkoutservice2-0'), np.str_('currencyservice-0'), np.str_('currencyservice-1'), np.str_('currencyservice-2'), np.str_('currencyservice2-0'), np.str_('emailservice-0'), np.str_('emailservice-1'), np.str_('emailservice-2'), np.str_('emailservice2-0')], 'kpi_trace_log_names_count': 200, 'kpi_trace_log_names_preview_up_to_50': [np.str_('trace.from_checkoutservice-0.duration_mean'), np.str_('trace.from_checkoutservice-0.duration_p95'), np.str_('trace.from_checkoutservice-0.error_rate'), np.str_('trace.from_checkoutservice-0.row_count'), np.str_('trace.from_checkoutservice-1.duration_mean'), np.str_('trace.from_checkoutservice-1.duration_p95'), np.str_('trace.from_checkoutservice-1.error_rate'), np.str_('trace.from_checkoutservice-1.row_count'), np.str_('trace.from_checkoutservice-2.duration_mean'), np.str_('trace.from_checkoutservice-2.duration_p95'), np.str_('trace.from_checkoutservice-2.error_rate'), np.str_('trace.from_checkoutservice-2.row_count'), np.str_('trace.from_checkoutservice2-0.duration_mean'), np.str_('trace.from_checkoutservice2-0.duration_p95'), np.str_('trace.from_checkoutservice2-0.error_rate'), np.str_('trace.from_checkoutservice2-0.row_count'), np.str_('trace.from_frontend-0.duration_mean'), np.str_('trace.from_frontend-0.duration_p95'), np.str_('trace.from_frontend-0.error_rate'), np.str_('trace.from_frontend-0.row_count'), np.str_('trace.from_frontend-1.duration_mean'), np.str_('trace.from_frontend-1.duration_p95'), np.str_('trace.from_frontend-1.error_rate'), np.str_('trace.from_frontend-1.row_count'), np.str_('trace.from_frontend-2.duration_mean'), np.str_('trace.from_frontend-2.duration_p95'), np.str_('trace.from_frontend-2.error_rate'), np.str_('trace.from_frontend-2.row_count'), np.str_('trace.from_frontend2-0.duration_mean'), np.str_('trace.from_frontend2-0.duration_p95'), np.str_('trace.from_frontend2-0.error_rate'), np.str_('trace.from_frontend2-0.row_count'), np.str_('trace.from_recommendationservice-0.duration_mean'), np.str_('trace.from_recommendationservice-0.duration_p95'), np.str_('trace.from_recommendationservice-0.error_rate'), np.str_('trace.from_recommendationservice-0.row_count'), np.str_('trace.from_recommendationservice-1.duration_mean'), np.str_('trace.from_recommendationservice-1.duration_p95'), np.str_('trace.from_recommendationservice-1.error_rate'), np.str_('trace.from_recommendationservice-1.row_count'), np.str_('trace.from_recommendationservice-2.duration_mean'), np.str_('trace.from_recommendationservice-2.duration_p95'), np.str_('trace.from_recommendationservice-2.error_rate'), np.str_('trace.from_recommendationservice-2.row_count'), np.str_('trace.from_recommendationservice2-0.duration_mean'), np.str_('trace.from_recommendationservice2-0.duration_p95'), np.str_('trace.from_recommendationservice2-0.error_rate'), np.str_('trace.from_recommendationservice2-0.row_count'), np.str_('trace.from_root.duration_mean'), np.str_('trace.from_root.duration_p95')], 'min_timestamp_as_Asia/Shanghai': '2022-03-20 16:30:00+0800', 'max_timestamp_as_Asia/Shanghai': '2022-03-20 16:59:00+0800'}, 'log.csv': {'total_rows': 1814, 'unique_cmdb_id_count': 31, 'unique_cmdb_id_preview_up_to_20': [np.str_('adservice-0'), np.str_('adservice-1'), np.str_('adservice-2'), np.str_('cartservice-0'), np.str_('cartservice-1'), np.str_('cartservice-2'), np.str_('checkoutservice-0'), np.str_('checkoutservice-1'), np.str_('checkoutservice-2'), np.str_('currencyservice-0'), np.str_('currencyservice-1'), np.str_('currencyservice-2'), np.str_('emailservice-0'), np.str_('emailservice-1'), np.str_('emailservice-2'), np.str_('frontend-0'), np.str_('frontend-1'), np.str_('frontend-2'), np.str_('paymentservice-0'), np.str_('paymentservice-1')], 'kpi_trace_log_names_count': 2, 'kpi_trace_log_names_preview_up_to_50': [np.str_('log.error_count'), np.str_('log.row_count')], 'min_timestamp_as_Asia/Shanghai': '2022-03-20 16:30:00+0800', 'max_timestamp_as_Asia/Shanghai': '2022-03-20 16:59:00+0800'}, 'error_logs.csv': {'total_rows': 2007, 'unique_cmdb_id_count': 3, 'unique_cmdb_id_preview_up_to_20': [np.str_('frontend-0'), np.str_('frontend-1'), np.str_('frontend-2')], 'message_column_present': True, 'message_samples_up_to_5_distinct': ['severity: warning, message: failed to retrieve ads'], 'min_timestamp_as_Asia/Shanghai': '2022-03-20 16:30:00+0800', 'max_timestamp_as_Asia/Shanghai': '2022-03-20 16:59:58+0800'}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure timestamp column as timezone-aware Asia/Shanghai datetimes (create/overwrite 'ts')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using all available rows
p95_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 2) Filter metric rows to incident window [2022-03-20 16:30:00+0800, 2022-03-20 17:00:00+0800)
start = pd.Timestamp('2022-03-20 16:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 17:00:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# 3) For each (cmdb_id, kpi_name) within window, count rows with value >= global P95, total rows, earliest breach ts, max value
# merge p95 into window
df_w = df_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# mark breaches
df_w['breach'] = df_w['value'] >= df_w['p95_value']

# group and aggregate with a apply to capture earliest breach timestamp
grouped = (
    df_w
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .apply(lambda g: pd.Series({
        'p95_value': float(g['p95_value'].iloc[0]) if pd.notna(g['p95_value'].iloc[0]) else float('nan'),
        'breach_count_in_window': int(g['breach'].sum()),
        'total_rows_in_window': int(g.shape[0]),
        'earliest_breach_timestamp': (g.loc[g['breach'], 'ts'].min() if g['breach'].any() else pd.NaT),
        'max_value_in_window': float(g['value'].max())
    }))
    .reset_index()
)

# format earliest_breach_timestamp as string with timezone, keep None/NaT handled
grouped['earliest_breach_timestamp'] = grouped['earliest_breach_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
grouped['earliest_breach_timestamp'] = grouped['earliest_breach_timestamp'].where(grouped['earliest_breach_timestamp'].notna(), None)

# 4) Return top 20 rows sorted by breach_count descending
result_df = (
    grouped
    .sort_values(by='breach_count_in_window', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'p95_value', 'breach_count_in_window', 'earliest_breach_timestamp', 'max_value_in_window', 'total_rows_in_window']]
    .head(20)
)

result_df

```
Out[2]:
```
summary = (
    "Top-20 results summary:\n\n"
    "All 20 top (cmdb_id, kpi_name) pairs are for cmdb_id 'shippingservice2-0' and relate to container.node-5 metrics. "
    "Each pair has breach_count_in_window = 30 (i.e., every minute in the 30-minute incident window met or exceeded its global P95), "
    "total_rows_in_window = 30, and the earliest breach occurred at 2022-03-20 16:30:00+0800. "
    "Example KPIs include container.node-5.container_ulimits_soft.max_op... (p95=65536), container.node-5.container_memory_cache (p95=18464768), "
    "container.node-5.container_memory_failcnt (p95=2397029), container.node-5.container_file_descriptors (p95=10), and many container_fs_* metrics with p95=0. "
    "Max values observed in the window match the reported p95 or shown max values for each pair."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name   p95_value  breach_count_in_window earliest_breach_timestamp  max_value_in_window  total_rows_in_window
4801  shippingservice2-0  container.node-5.container_ulimits_soft.max_op...     65536.0                      30  2022-03-20 16:30:00+0800              65536.0                    30
4761  shippingservice2-0      container.node-5.container_fs_writes./dev/vda         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4762  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4763  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4764  shippingservice2-0  container.node-5.container_fs_writes_merged./d...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4766  shippingservice2-0            container.node-5.container_memory_cache  18464768.0                      30  2022-03-20 16:30:00+0800           18464768.0                    30
4767  shippingservice2-0          container.node-5.container_memory_failcnt   2397029.0                      30  2022-03-20 16:30:00+0800            2397029.0                    30
4753  shippingservice2-0       container.node-5.container_fs_reads./dev/vda         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4754  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4755  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4756  shippingservice2-0  container.node-5.container_fs_reads_merged./de...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4757  shippingservice2-0  container.node-5.container_fs_sector_reads./de...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4758  shippingservice2-0  container.node-5.container_fs_sector_writes./d...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4760  shippingservice2-0  container.node-5.container_fs_write_seconds./d...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4745  shippingservice2-0        container.node-5.container_file_descriptors        10.0                      30  2022-03-20 16:30:00+0800                 10.0                    30
4746  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4747  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4748  shippingservice2-0  container.node-5.container_fs_io_current./dev/...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4749  shippingservice2-0  container.node-5.container_fs_io_time_seconds....         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30
4750  shippingservice2-0  container.node-5.container_fs_io_time_weighted...         0.0                      30  2022-03-20 16:30:00+0800                  0.0                    30```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse tz and df_trace if available
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure timezone-aware timestamp column 'ts' in Asia/Shanghai
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) select trace_name entries containing 'shippingservice2-0' OR 'shippingservice-'
mask_select = df_trace['trace_name'].astype(str).str.contains('shippingservice2-0') | df_trace['trace_name'].astype(str).str.contains('shippingservice-')
selected_trace_names = df_trace.loc[mask_select, 'trace_name'].dropna().unique()
selected_trace_names.sort()

# 2) compute global P95 per trace_name (using all rows for that trace_name)
p95_trace = (
    df_trace
    .loc[df_trace['trace_name'].isin(selected_trace_names), :]
    .groupby('trace_name', observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 3) filter trace rows to incident window [2022-03-20 16:30:00+0800, 2022-03-20 17:00:00+0800)
start = pd.Timestamp('2022-03-20 16:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 17:00:00', tz=tz)
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()

# keep only selected trace_names in window
df_tw_sel = df_trace_window[df_trace_window['trace_name'].isin(selected_trace_names)].copy()

# 4) merge p95 and compute breach metrics per trace_name in window
df_tw_sel = df_tw_sel.merge(p95_trace, on='trace_name', how='left')
df_tw_sel['breach'] = df_tw_sel['value'] >= df_tw_sel['p95_value']

agg = (
    df_tw_sel
    .groupby('trace_name', observed=True)
    .apply(lambda g: pd.Series({
        'p95_value': float(g['p95_value'].iloc[0]) if pd.notna(g['p95_value'].iloc[0]) else float('nan'),
        'breach_count_in_window': int(g['breach'].sum()),
        'earliest_breach_timestamp': (g.loc[g['breach'], 'ts'].min() if g['breach'].any() else pd.NaT),
        'max_value_in_window': float(g['value'].max()),
        'total_rows_in_window': int(g.shape[0])
    }))
    .reset_index()
)

# format earliest_breach_timestamp as string with timezone, keep None for NaT
agg['earliest_breach_timestamp'] = agg['earliest_breach_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg['earliest_breach_timestamp'] = agg['earliest_breach_timestamp'].where(agg['earliest_breach_timestamp'].notna(), None)

# 5) return up to top 20 trace_name rows sorted by breach_count_in_window desc
result_trace_top20 = (
    agg
    .sort_values(by='breach_count_in_window', ascending=False)
    .loc[:, ['trace_name', 'p95_value', 'breach_count_in_window', 'earliest_breach_timestamp', 'max_value_in_window', 'total_rows_in_window']]
    .head(20)
)

result_trace_top20

```
Out[3]:
```
Summary of results (top trace_name rows by breach_count within 2022-03-20 16:30:00+08 to 17:00:00+08):

- trace.to_shippingservice-0.error_rate — p95=0.0, breach_count=180, earliest_breach=2022-03-20 16:30:00+0800, max=0.0, total_rows=180
- trace.to_shippingservice-1.error_rate — p95=0.0, breach_count=180, earliest_breach=2022-03-20 16:30:00+0800, max=0.0, total_rows=180
- trace.to_shippingservice-2.error_rate — p95=0.0, breach_count=180, earliest_breach=2022-03-20 16:30:00+0800, max=0.0, total_rows=180
- trace.to_shippingservice2-0.error_rate — p95=0.0, breach_count=60,  earliest_breach=2022-03-20 16:30:00+0800, max=0.0, total_rows=60
- trace.to_shippingservice-2.duration_mean — p95=0.000068, breach_count=11, earliest_breach=2022-03-20 16:31:00+0800, max=0.000142, total_rows=180
- trace.to_shippingservice-2.row_count — p95=11.0, breach_count=10, earliest_breach=2022-03-20 16:38:00+0800, max=12.0, total_rows=180
- trace.to_shippingservice-0.row_count — p95=11.0, breach_count=10, earliest_breach=2022-03-20 16:32:00+0800, max=12.0, total_rows=180
- trace.to_shippingservice-1.row_count — p95=10.05, breach_count=9, earliest_breach=2022-03-20 16:38:00+0800, max=12.0, total_rows=180
- trace.to_shippingservice-0.duration_p95 — p95=0.000079, breach_count=9, earliest_breach=2022-03-20 16:30:00+0800, max=0.000361, total_rows=180
- trace.to_shippingservice-1.duration_p95 — p95=0.000082, breach_count=9, earliest_breach=2022-03-20 16:30:00+0800, max=0.000359, total_rows=180
- trace.to_shippingservice-1.duration_mean — p95=0.000067, breach_count=9, earliest_breach=2022-03-20 16:30:00+0800, max=0.000159, total_rows=180
- trace.to_shippingservice-0.duration_mean — p95=0.000065, breach_count=9, earliest_breach=2022-03-20 16:30:00+0800, max=0.000186, total_rows=180
- trace.to_shippingservice-2.duration_p95 — p95=0.000085, breach_count=9, earliest_breach=2022-03-20 16:31:00+0800, max=0.000355, total_rows=180
- trace.to_shippingservice2-0.duration_mean — p95=0.000073, breach_count=4, earliest_breach=2022-03-20 16:43:00+0800, max=0.000078, total_rows=60
- trace.to_shippingservice2-0.row_count — p95=19.0, breach_count=4, earliest_breach=2022-03-20 16:33:00+0800, max=22.0, total_rows=60
- trace.to_shippingservice2-0.duration_p95 — p95=0.000120, breach_count=3, earliest_breach=2022-03-20 16:47:00+0800, max=0.000151, total_rows=60

Key takeaway: the highest breach counts are the shippingservice error_rate traces (three main pods had every row in the window counted as breach), followed by various duration and row_count traces with fewer breaches.

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  p95_value  breach_count_in_window earliest_breach_timestamp  max_value_in_window  total_rows_in_window
2       trace.to_shippingservice-0.error_rate   0.000000                     180  2022-03-20 16:30:00+0800             0.000000                   180
6       trace.to_shippingservice-1.error_rate   0.000000                     180  2022-03-20 16:30:00+0800             0.000000                   180
10      trace.to_shippingservice-2.error_rate   0.000000                     180  2022-03-20 16:30:00+0800             0.000000                   180
14     trace.to_shippingservice2-0.error_rate   0.000000                      60  2022-03-20 16:30:00+0800             0.000000                    60
8    trace.to_shippingservice-2.duration_mean   0.000068                      11  2022-03-20 16:31:00+0800             0.000142                   180
11       trace.to_shippingservice-2.row_count  11.000000                      10  2022-03-20 16:38:00+0800            12.000000                   180
3        trace.to_shippingservice-0.row_count  11.000000                      10  2022-03-20 16:32:00+0800            12.000000                   180
1     trace.to_shippingservice-0.duration_p95   0.000079                       9  2022-03-20 16:30:00+0800             0.000361                   180
7        trace.to_shippingservice-1.row_count  10.050000                       9  2022-03-20 16:38:00+0800            12.000000                   180
5     trace.to_shippingservice-1.duration_p95   0.000082                       9  2022-03-20 16:30:00+0800             0.000359                   180
4    trace.to_shippingservice-1.duration_mean   0.000067                       9  2022-03-20 16:30:00+0800             0.000159                   180
0    trace.to_shippingservice-0.duration_mean   0.000065                       9  2022-03-20 16:30:00+0800             0.000186                   180
9     trace.to_shippingservice-2.duration_p95   0.000085                       9  2022-03-20 16:31:00+0800             0.000355                   180
12  trace.to_shippingservice2-0.duration_mean   0.000073                       4  2022-03-20 16:43:00+0800             0.000078                    60
15      trace.to_shippingservice2-0.row_count  19.000000                       4  2022-03-20 16:33:00+0800            22.000000                    60
13   trace.to_shippingservice2-0.duration_p95   0.000120                       3  2022-03-20 16:47:00+0800             0.000151                    60```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing variables if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column 'ts'
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) select rows where kpi_name contains 'container.node-5' (use full dataset for global P95 calculation)
mask_cn5 = df_metric['kpi_name'].astype(str).str.contains('container.node-5', na=False)
df_cn5_all = df_metric[mask_cn5].copy()

# 2) compute global P95 per (cmdb_id, kpi_name) using all available rows for that pair
p95_cn5 = (
    df_cn5_all
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 3) filter rows to the incident window [2022-03-20 16:30:00+0800, 2022-03-20 17:00:00+0800)
start = pd.Timestamp('2022-03-20 16:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 17:00:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# keep only container.node-5 rows in the window
df_window_cn5 = df_window[ df_window['kpi_name'].astype(str).str.contains('container.node-5', na=False) ].copy()

# 4) merge global p95 into window rows and compute breach flags
df_window_cn5 = df_window_cn5.merge(p95_cn5, on=['cmdb_id', 'kpi_name'], how='left')
df_window_cn5['breach'] = df_window_cn5['value'] >= df_window_cn5['p95_value']

# aggregate per (cmdb_id, kpi_name) within the window
per_pair = (
    df_window_cn5
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        p95_value = ('p95_value', lambda s: float(s.iloc[0]) if pd.notna(s.iloc[0]) else float('nan')),
        breach_count_in_window = ('breach', 'sum'),
        total_rows_in_window = ('value', 'count'),
        earliest_breach_ts = ('ts', lambda s: s[s.index.isin(s[s.index[s.index.isin(s.index)]] )].loc[df_window_cn5.loc[s.index, 'breach']].min() if True else pd.NaT)
    )
    .reset_index()
)

# The above earliest_breach_ts agg is complex due to lambda; simpler compute earliest breach timestamp explicitly:
# recompute earliest_breach_ts correctly:
breach_times = df_window_cn5[df_window_cn5['breach']].groupby(['cmdb_id','kpi_name'], observed=True)['ts'].min().reset_index().rename(columns={'ts':'earliest_breach_ts'})
per_pair = per_pair.drop(columns=['earliest_breach_ts']).merge(breach_times, on=['cmdb_id','kpi_name'], how='left')

# 5) aggregate per cmdb_id
# num_kpis_total = count of distinct kpi_name present in the window for that cmdb_id (i.e., per_pair rows)
# num_kpis_fully_breached = count where breach_count_in_window == total_rows_in_window and total_rows_in_window>0
# total_breaches_in_window = sum of breach_count_in_window
# earliest_breach_timestamp_across_kpis = min timestamp among breaches for that cmdb_id
per_cmdb = (
    per_pair
    .groupby('cmdb_id', observed=True)
    .apply(lambda g: pd.Series({
        'num_kpis_total': int(g['kpi_name'].nunique()),
        'num_kpis_fully_breached': int(((g['breach_count_in_window'] == g['total_rows_in_window']) & (g['total_rows_in_window']>0)).sum()),
        'total_breaches_in_window': int(g['breach_count_in_window'].sum()),
        'earliest_breach_timestamp_across_kpis': (g['earliest_breach_ts'].min() if g['earliest_breach_ts'].notna().any() else pd.NaT)
    }))
    .reset_index()
)

# format earliest timestamp as string with timezone, replace NaT with None
per_cmdb['earliest_breach_timestamp_across_kpis'] = per_cmdb['earliest_breach_timestamp_across_kpis'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
per_cmdb['earliest_breach_timestamp_across_kpis'] = per_cmdb['earliest_breach_timestamp_across_kpis'].where(per_cmdb['earliest_breach_timestamp_across_kpis'].notna(), None)

# 6) return top 20 cmdb_id rows sorted by num_kpis_fully_breached descending
result_cmdb_top20 = per_cmdb.sort_values(by='num_kpis_fully_breached', ascending=False).loc[:, [
    'cmdb_id',
    'num_kpis_total',
    'num_kpis_fully_breached',
    'total_breaches_in_window',
    'earliest_breach_timestamp_across_kpis'
]].head(20)

result_cmdb_top20

```
Out[4]:
```
summary = (
    "Summary of container.node-5 KPIs during the incident window (2022-03-20 16:30 to 17:00 +08):\n\n"
    "Top affected cmdb_id by number of fully-breached KPIs (a KPI is 'fully breached' if every sample in the window >= its global P95):\n"
    "1) emailservice-2 — num_kpis_total=64, num_kpis_fully_breached=54, total_breaches_in_window=1654, earliest_breach=2022-03-20 16:30:00+0800\n"
    "2) emailservice-0 — num_kpis_total=64, num_kpis_fully_breached=54, total_breaches_in_window=1646, earliest_breach=2022-03-20 16:30:00+0800\n"
    "3) emailservice-1 — num_kpis_total=64, num_kpis_fully_breached=52, total_breaches_in_window=1603, earliest_breach=2022-03-20 16:30:00+0800\n"
    "4) emailservice2-0 — num_kpis_total=64, num_kpis_fully_breached=52, total_breaches_in_window=1641, earliest_breach=2022-03-20 16:30:00+0800\n"
    "5) recommendationservice-1 — num_kpis_total=64, num_kpis_fully_breached=51, total_breaches_in_window=1557, earliest_breach=2022-03-20 16:30:00+0800\n"
    "6) recommendationservice-0 — num_kpis_total=64, num_kpis_fully_breached=51, total_breaches_in_window=1613, earliest_breach=2022-03-20 16:30:00+0800\n"
    "7) recommendationservice-2 — num_kpis_total=64, num_kpis_fully_breached=51, total_breaches_in_window=1577, earliest_breach=2022-03-20 16:30:00+0800\n"
    "8) productcatalogservice-0 — num_kpis_total=64, num_kpis_fully_breached=50, total_breaches_in_window=1528, earliest_breach=2022-03-20 16:30:00+0800\n"
    "9) paymentservice-2 — num_kpis_total=64, num_kpis_fully_breached=49, total_breaches_in_window=1501, earliest_breach=2022-03-20 16:30:00+0800\n"
    "10) shippingservice-2 — num_kpis_total=64, num_kpis_fully_breached=49, total_breaches_in_window=1505, earliest_breach=2022-03-20 16:30:00+0800\n\n"
    "Overall takeaway: many services (notably multiple emailservice, recommendationservice, productcatalogservice, paymentservice, and shippingservice pods) had a large number of container.node-5 KPIs fully breached throughout the 30-minute window, with the earliest breaches starting at 2022-03-20 16:30:00+0800. These services are the top candidates for investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  num_kpis_total  num_kpis_fully_breached  total_breaches_in_window earliest_breach_timestamp_across_kpis
15           emailservice-2              64                       54                      1654              2022-03-20 16:30:00+0800
13           emailservice-0              64                       54                      1646              2022-03-20 16:30:00+0800
14           emailservice-1              64                       52                      1603              2022-03-20 16:30:00+0800
16          emailservice2-0              64                       52                      1641              2022-03-20 16:30:00+0800
27  recommendationservice-1              64                       51                      1557              2022-03-20 16:30:00+0800
26  recommendationservice-0              64                       51                      1613              2022-03-20 16:30:00+0800
28  recommendationservice-2              64                       51                      1577              2022-03-20 16:30:00+0800
23  productcatalogservice-0              64                       50                      1528              2022-03-20 16:30:00+0800
22         paymentservice-2              64                       49                      1501              2022-03-20 16:30:00+0800
33        shippingservice-2              64                       49                      1505              2022-03-20 16:30:00+0800
20         paymentservice-0              64                       49                      1504              2022-03-20 16:30:00+0800
10        currencyservice-0              64                       49                      1501              2022-03-20 16:30:00+0800
11        currencyservice-1              64                       49                      1500              2022-03-20 16:30:00+0800
12        currencyservice-2              64                       49                      1502              2022-03-20 16:30:00+0800
31        shippingservice-0              64                       49                      1500              2022-03-20 16:30:00+0800
34       shippingservice2-0              64                       49                      1506              2022-03-20 16:30:00+0800
25  productcatalogservice-2              64                       48                      1472              2022-03-20 16:30:00+0800
24  productcatalogservice-1              64                       48                      1474              2022-03-20 16:30:00+0800
32        shippingservice-1              64                       48                      1473              2022-03-20 16:30:00+0800
7         checkoutservice-1              64                       47                      1447              2022-03-20 16:30:00+0800```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse existing df_metric and tz if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column 'ts'
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) select rows where cmdb_id == 'node-5' OR kpi_name contains 'node-5'
mask_selection = (df_metric['cmdb_id'] == 'node-5') | df_metric['kpi_name'].astype(str).str.contains('node-5', na=False)
selected_kpi_names = pd.Series(df_metric.loc[mask_selection, 'kpi_name'].dropna().unique()).sort_values().reset_index(drop=True)

# 2) compute global P95 per kpi_name using the full series (all rows for each kpi_name)
p95_by_kpi = df_metric.groupby('kpi_name', observed=True)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95_value'})

# keep only p95 for selected kpis
p95_selected = p95_by_kpi[p95_by_kpi['kpi_name'].isin(selected_kpi_names)]

# 3) filter selection rows to the incident window [2022-03-20 16:30:00+0800, 2022-03-20 17:00:00+0800)
start = pd.Timestamp('2022-03-20 16:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 17:00:00', tz=tz)
df_window_selection = df_metric[mask_selection & (df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# 4) merge p95 and compute metrics per kpi_name in the window
df_window_sel = df_window_selection.merge(p95_selected, on='kpi_name', how='left')
df_window_sel['breach'] = df_window_sel['value'] >= df_window_sel['p95_value']

agg_kpi = (
    df_window_sel
    .groupby('kpi_name', observed=True)
    .agg(
        p95_value = ('p95_value', lambda s: float(s.iloc[0]) if pd.notna(s.iloc[0]) else float('nan')),
        breach_count_in_window = ('breach', 'sum'),
        earliest_breach_timestamp = ('ts', lambda s: s.loc[df_window_sel.loc[s.index, 'breach']].min() if (df_window_sel.loc[s.index, 'breach'].any()) else pd.NaT),
        max_value_in_window = ('value', 'max'),
        total_rows_in_window = ('value', 'count')
    )
    .reset_index()
)

# format earliest breach timestamp
agg_kpi['earliest_breach_timestamp'] = agg_kpi['earliest_breach_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg_kpi['earliest_breach_timestamp'] = agg_kpi['earliest_breach_timestamp'].where(agg_kpi['earliest_breach_timestamp'].notna(), None)

# 5) return top 20 kpi_name rows sorted by breach_count_in_window desc
result_kpi_top20 = (
    agg_kpi
    .sort_values(by='breach_count_in_window', ascending=False)
    .loc[:, ['kpi_name', 'p95_value', 'breach_count_in_window', 'earliest_breach_timestamp', 'max_value_in_window', 'total_rows_in_window']]
    .head(20)
)

result_kpi_top20

```
Out[5]:
```
Summary of results (top kpi_names for node-5 during 2022-03-20 16:30–17:00 +08):

- The highest breach counts are for many container.node-5 metrics (network, fs, cpu, tasks, memory/failure counters). Examples at the top:
  - container.node-5.container_network_transmit_packets... — p95=0.0, breach_count=1050, max=0.0, total_rows=1050, earliest breach=2022-03-20 16:30:00+0800
  - container.node-5.container_network_receive_errors... — p95=0.0, breach_count=1050, max=0.0, total_rows=1050, earliest breach=2022-03-20 16:30:00+0800
  - container.node-5.container_network_receive_packets... — p95=0.0, breach_count=1050, max=0.0, total_rows=1050, earliest breach=2022-03-20 16:30:00+0800
  - container.node-5.container_network_transmit_errors... — p95=0.0, breach_count=1050, max=0.0, total_rows=1050, earliest breach=2022-03-20 16:30:00+0800
  - Many fs-related metrics (reads, read_seconds, inodes, sector_reads, etc.) show p95=0.0 with breach_count=990, earliest breach=2022-03-20 16:30:00+0800.
  - One notable non-zero example: container.node-5.container_spec_cpu_period — p95=100000.0, breach_count=990, max=100000.0, total_rows=990, earliest breach=2022-03-20 16:30:00+0800.

Interpretation: a large set of node-5 related container KPIs were at or above their global P95 throughout the incident window (many with every sample counted as a breach). Note that many p95 values are 0.0, so breaches for those metrics mean the window values were >=0 (often also 0), which can reflect that the baseline distribution is sparse/mostly-zero and should be interpreted cautiously. The earliest breaches across these top KPIs all occur at the start of the window (2022-03-20 16:30:00+0800).

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  p95_value  breach_count_in_window earliest_breach_timestamp  max_value_in_window  total_rows_in_window
47  container.node-5.container_network_transmit_pa...        0.0                    1050  2022-03-20 16:30:00+0800                  0.0                  1050
41  container.node-5.container_network_receive_err...        0.0                    1050  2022-03-20 16:30:00+0800                  0.0                  1050
43  container.node-5.container_network_receive_pac...        0.0                    1050  2022-03-20 16:30:00+0800                  0.0                  1050
45  container.node-5.container_network_transmit_er...        0.0                    1050  2022-03-20 16:30:00+0800                  0.0                  1050
16      container.node-5.container_fs_reads./dev/vda1        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
14  container.node-5.container_fs_read_seconds./de...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
8      container.node-5.container_fs_inodes./dev/vda1        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
19  container.node-5.container_fs_sector_reads./de...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
9   container.node-5.container_fs_inodes_free./dev...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
3     container.node-5.container_cpu_load_average_10s        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
24     container.node-5.container_fs_writes./dev/vda1        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
22  container.node-5.container_fs_write_seconds./d...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
20  container.node-5.container_fs_sector_writes./d...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
18  container.node-5.container_fs_reads_merged./de...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
57     container.node-5.container_tasks_state.running        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
56   container.node-5.container_tasks_state.iowaiting        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
53  container.node-5.container_spec_memory_reserva...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
49         container.node-5.container_spec_cpu_period   100000.0                     990  2022-03-20 16:30:00+0800             100000.0                   990
31  container.node-5.container_memory_failures.con...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990
33  container.node-5.container_memory_failures.hie...        0.0                     990  2022-03-20 16:30:00+0800                  0.0                   990```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse existing df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column 'ts'
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# selection: kpi_name contains 'node-5' OR cmdb_id == 'node-5'
mask_node5 = df_metric['kpi_name'].astype(str).str.contains('node-5', na=False) | (df_metric['cmdb_id'] == 'node-5')
df_node5 = df_metric[mask_node5].copy()

# distinct kpi names (sorted) and preview up to 100
distinct_kpis = sorted(df_node5['kpi_name'].dropna().unique().astype(str))
distinct_kpi_count = len(distinct_kpis)
kpi_preview = distinct_kpis[:100]

# total matching rows and min/max timestamps (Asia/Shanghai)
total_rows_matching = int(df_node5.shape[0])
if total_rows_matching > 0:
    tmin = df_node5['ts'].min().strftime('%Y-%m-%d %H:%M:%S%z')
    tmax = df_node5['ts'].max().strftime('%Y-%m-%d %H:%M:%S%z')
else:
    tmin = None
    tmax = None

# compact summary as a single-row DataFrame
result_inventory = pd.DataFrame([{
    'distinct_kpi_count': distinct_kpi_count,
    'total_rows_matching_filter': total_rows_matching,
    'min_timestamp_as_Asia/Shanghai': tmin,
    'max_timestamp_as_Asia/Shanghai': tmax,
    'kpi_preview_up_to_100': kpi_preview
}])

result_inventory

```
Out[6]:
```
import pandas as pd
import pytz

# try to reuse previous summary variable; otherwise recompute
try:
    info = result_inventory.iloc[0]
    distinct_count = int(info['distinct_kpi_count'])
    total_rows = int(info['total_rows_matching_filter'])
    tmin = info['min_timestamp_as_Asia/Shanghai']
    tmax = info['max_timestamp_as_Asia/Shanghai']
    kpi_preview_full = info['kpi_preview_up_to_100']
except Exception:
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    mask_node5 = df_metric['kpi_name'].astype(str).str.contains('node-5', na=False) | (df_metric['cmdb_id'] == 'node-5')
    df_node5 = df_metric[mask_node5].copy()
    distinct_kpis = sorted(df_node5['kpi_name'].dropna().unique().astype(str))
    distinct_count = len(distinct_kpis)
    total_rows = int(df_node5.shape[0])
    if total_rows > 0:
        tmin = df_node5['ts'].min().strftime('%Y-%m-%d %H:%M:%S%z')
        tmax = df_node5['ts'].max().strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        tmin = tmax = None
    kpi_preview_full = distinct_kpis[:100]

# prepare a short preview (up to 20) for compact display
kpi_preview_short = kpi_preview_full[:20]

summary = (
    f"Node-5 KPI inventory:\n"
    f"- Distinct kpi_name count: {distinct_count}\n"
    f"- Total matching rows: {total_rows:,}\n"
    f"- Time range (Asia/Shanghai): {tmin} to {tmax}\n"
    f"- Preview (up to 20 of {distinct_count}, sorted): {kpi_preview_short}\n"
    f"(Full preview up to 100 available in variable 'kpi_preview_full')"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

distinct_kpi_count  total_rows_matching_filter min_timestamp_as_Asia/Shanghai max_timestamp_as_Asia/Shanghai                              kpi_preview_up_to_100
0                 122                       64932       2022-03-20 16:30:00+0800       2022-03-20 16:59:00+0800  [container.node-5.container_cpu_cfs_periods, c...```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse existing variables from the session when possible
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column 'ts'
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) selection mask: kpi_name contains 'node-5' OR cmdb_id == 'node-5'
mask_node5 = df_metric['kpi_name'].astype(str).str.contains('node-5', na=False) | (df_metric['cmdb_id'] == 'node-5')
selected_kpis = sorted(df_metric.loc[mask_node5, 'kpi_name'].dropna().unique().astype(str))

# 2) compute global P95 per kpi_name using the entire available series (global, before window filtering)
p95_by_kpi = (
    df_metric
    .groupby('kpi_name', observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# keep only p95 rows for the selected kpis
p95_selected = p95_by_kpi[p95_by_kpi['kpi_name'].isin(selected_kpis)].copy()

# 3) filter rows to the incident window [2022-03-20 16:30:00+0800, 2022-03-20 17:00:00+0800)
start = pd.Timestamp('2022-03-20 16:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 17:00:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# keep only rows matching our selection within the window
df_window_sel = df_window[mask_node5.loc[df_window.index].values].copy()

# 4) merge p95 into windowed selection and compute metrics per kpi_name
df_w = df_window_sel.merge(p95_selected, on='kpi_name', how='left')
df_w['breach'] = df_w['value'] >= df_w['p95_value']

agg = (
    df_w
    .groupby('kpi_name', observed=True)
    .agg(
        p95_value = ('p95_value', lambda s: float(s.iloc[0]) if pd.notna(s.iloc[0]) else float('nan')),
        max_value_in_window = ('value', 'max'),
        breach_count_in_window = ('breach', 'sum'),
        total_rows_in_window = ('value', 'count'),
    )
    .reset_index()
)

# earliest breach timestamp per kpi_name
earliest = (
    df_w[df_w['breach']]
    .groupby('kpi_name', observed=True)['ts']
    .min()
    .reset_index()
    .rename(columns={'ts': 'earliest_breach_timestamp'})
)

agg = agg.merge(earliest, on='kpi_name', how='left')

# format earliest timestamp and compute severity
agg['earliest_breach_timestamp'] = agg['earliest_breach_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
agg['earliest_breach_timestamp'] = agg['earliest_breach_timestamp'].where(agg['earliest_breach_timestamp'].notna(), None)

# severity: if p95_value > 0 -> max_value / p95_value else severity = max_value
def compute_severity(row):
    try:
        p95 = float(row['p95_value'])
    except Exception:
        return float('nan')
    maxv = float(row['max_value_in_window']) if pd.notna(row['max_value_in_window']) else float('nan')
    if pd.isna(p95) or p95 == 0:
        return maxv
    else:
        # avoid division by zero, p95>0 handled
        return maxv / p95

agg['severity'] = agg.apply(compute_severity, axis=1)

# 5) sort by severity desc and return top 20 with requested columns
result_kpi_severity_top20 = (
    agg
    .sort_values(by='severity', ascending=False)
    .loc[:, ['kpi_name', 'p95_value', 'max_value_in_window', 'severity', 'breach_count_in_window', 'total_rows_in_window', 'earliest_breach_timestamp']]
    .head(20)
)

result_kpi_severity_top20

```
Out[7]:
```
Summary — top 20 node-5 related KPIs by severity (2022-03-20 16:30–17:00 +08):

Top highlights (kpi_name — p95_value, max_in_window, severity, breach_count, total_rows, earliest_breach):

1. container.node-5.container_memory_max_usage_MB — p95=256.01, max=1025.00, severity=4.00, breaches=60 / 990, earliest=2022-03-20 16:30:00+0800  
2. container.node-5.container_cpu_cfs_throttled_seconds… — p95=0.0937, max=0.3145, severity=3.36, breaches=50 / 990, earliest=2022-03-20 16:30:00+0800  
3. container.node-5.container_memory_failcnt — p95=2,990,153, max=9,628,125, severity=3.22, breaches=60 / 990, earliest=2022-03-20 16:30:00+0800  
4. container.node-5.container_cpu_cfs_throttled_periods — p95=1.50, max=4.00, severity=2.67, breaches=81 / 990, earliest=2022-03-20 16:30:00+0800  
5. system.io.await — p95=1.4145, max=3.5600, severity=2.52, breaches=8 / 30, earliest=2022-03-20 16:30:00+0800  
6. system.io.w_await — p95=1.6920, max=3.5600, severity=2.10, breaches=7 / 30, earliest=2022-03-20 16:30:00+0800  
7. system.io.svctm — p95=0.4005, max=0.7800, severity=1.95, breaches=1 / 30, earliest=2022-03-20 16:30:00+0800  
8. container.node-5.container_network_receive_MB… — p95=1.2697, max=1.9349, severity=1.52, breaches=53 / 1050, earliest=2022-03-20 16:30:00+0800  
9. container.node-5.container_network_transmit_MB… — p95=4.8519, max=6.6942, severity=1.38, breaches=53 / 1050, earliest=2022-03-20 16:30:00+0800  
10. container.node-5.container_memory_failures.container.pgfault (and hierarchy) — p95≈3762.78, max≈5111.00, severity≈1.36, breaches=50 / 990, earliest=2022-03-20 16:30:00+0800

(Additional top entries include container memory usage, CPU system/user/usage seconds, network packet counters, and system.io.w_s; all top-20 rows show severity >1 and earliest breaches at or near 16:30+08.)

Straightforward interpretation:
- Several node-5 container memory and CPU throttling metrics show the largest severity (memory max usage ≈4× P95; CPU throttling and memory fail counters ≈3× P95), indicating notable resource pressure on node-5 during the incident window.  
- Network and filesystem metrics also exceed P95 but with lower severity (≈1.2–1.5×) and large sample counts.  
- Many earliest breaches occur at the start of the window (2022-03-20 16:30:00+0800).  
- Note: some p95 values are small or zero for certain metrics; severity is computed so that non-zero large max values rank high even when p95 is zero.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name     p95_value  max_value_in_window  severity  breach_count_in_window  total_rows_in_window earliest_breach_timestamp
35     container.node-5.container_memory_max_usage_MB  2.560078e+02         1.025000e+03  4.003784                      60                   990  2022-03-20 16:30:00+0800
2   container.node-5.container_cpu_cfs_throttled_s...  9.369099e-02         3.144702e-01  3.356461                      50                   990  2022-03-20 16:30:00+0800
29          container.node-5.container_memory_failcnt  2.990153e+06         9.628125e+06  3.219944                      60                   990  2022-03-20 16:30:00+0800
1   container.node-5.container_cpu_cfs_throttled_p...  1.500000e+00         4.000000e+00  2.666667                      81                   990  2022-03-20 16:30:00+0800
78                                    system.io.await  1.414500e+00         3.560000e+00  2.516790                       8                    30  2022-03-20 16:30:00+0800
84                                  system.io.w_await  1.692000e+00         3.560000e+00  2.104019                       7                    30  2022-03-20 16:30:00+0800
82                                    system.io.svctm  4.005000e-01         7.800000e-01  1.947566                       1                    30  2022-03-20 16:30:00+0800
40  container.node-5.container_network_receive_MB....  1.269738e+00         1.934891e+00  1.523851                      53                  1050  2022-03-20 16:30:00+0800
44  container.node-5.container_network_transmit_MB...  4.851869e+00         6.694231e+00  1.379722                      53                  1050  2022-03-20 16:30:00+0800
30  container.node-5.container_memory_failures.con...  3.762775e+03         5.111000e+03  1.358306                      50                   990  2022-03-20 16:30:00+0800
32  container.node-5.container_memory_failures.hie...  3.762775e+03         5.111000e+03  1.358306                      50                   990  2022-03-20 16:30:00+0800
38         container.node-5.container_memory_usage_MB  1.593446e+02         2.046660e+02  1.284424                      50                   990  2022-03-20 16:30:00+0800
4       container.node-5.container_cpu_system_seconds  3.800000e-01         4.800000e-01  1.263158                      50                   990  2022-03-20 16:30:00+0800
46  container.node-5.container_network_transmit_pa...  6.905450e+03         8.513500e+03  1.232867                      53                  1050  2022-03-20 16:30:00+0800
0          container.node-5.container_cpu_cfs_periods  3.820500e+02         4.665000e+02  1.221044                      50                   990  2022-03-20 16:30:00+0800
6         container.node-5.container_cpu_user_seconds  4.277500e-01         5.200000e-01  1.215663                      50                   990  2022-03-20 16:30:00+0800
42  container.node-5.container_network_receive_pac...  6.838075e+03         8.274500e+03  1.210063                      53                  1050  2022-03-20 16:30:00+0800
5        container.node-5.container_cpu_usage_seconds  9.757485e-01         1.158519e+00  1.187313                      50                   990  2022-03-20 16:30:00+0800
85                                      system.io.w_s  4.542350e+01         5.347000e+01  1.177144                       1                    30  2022-03-20 16:35:00+0800
34      container.node-5.container_memory_mapped_file  1.656422e+07         1.881293e+07  1.135757                      60                   990  2022-03-20 16:30:00+0800```
```